# ML Zoomcamp 2025 – Module 9: Serverless Hair Type Classifier

This notebook walks through the **Module 9 (Serverless)** homework for the **Straight vs Curly Hair Type** model.

It is based on the model trained in **Module 8** and shows how to:

1. Load and inspect the ONNX model (input/output node names).
2. Download and preprocess an image **exactly** like in HW8 (resize, scale, normalize).
3. Run inference with **onnxruntime** and get the prediction.
4. Wrap the logic into a reusable `predict_from_url` function.
5. Build a Lambda-style handler for deployment.
6. Outline the Docker steps for running the model serverlessly.

> ⚠️ **Note:** Do NOT run this notebook in the ML Zoomcamp grader. Run it locally (or in Colab) where you can install extra libraries like `onnx` and `onnxruntime`.


## 0. Setup and prerequisites

You will need these files in the same directory as this notebook:

- `hair_classifier_v1.onnx`
- `hair_classifier_v1.onnx.data` (weight data for ONNX)

You also need the following Python packages:

```bash
pip install pillow numpy onnx onnxruntime
```


## 1. Question 1 – Inspect the ONNX model (input & output names)

Here we load `hair_classifier_v1.onnx` and print:

- Names and shapes of all **inputs**
- Names and shapes of all **outputs**

This is how we confirm the **output node name** used in Question 1.


In [4]:
import onnx

model_path = "hair_classifier_v1.onnx"

model = onnx.load(model_path)
graph = model.graph

print("=== Inputs ===")
for inp in graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"Name: {inp.name}, Shape: {shape}")

print("\n=== Outputs ===")
for out in graph.output:
    shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"Name: {out.name}, Shape: {shape}")

=== Inputs ===
Name: input, Shape: [0, 3, 200, 200]

=== Outputs ===
Name: output, Shape: [0, 1]


## 2. Question 2 & 3 – Download and preprocess the image

In Module 8, we used the following transforms:

```python
transforms.Resize((200, 200)),
transforms.ToTensor(),
transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)
```

We now replicate this behavior using **PIL + NumPy**:

1. Download the image with `urllib.request`.
2. Resize it to **(200, 200)** (answer to Question 2).
3. Convert to a NumPy array and scale to `[0, 1]`.
4. Apply ImageNet normalization.
5. Print the value of the **first pixel, R channel, after normalization** (Question 3).


In [5]:
from io import BytesIO
from urllib import request
from PIL import Image
import numpy as np

IMG_URL = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"

def download_image(url: str) -> Image.Image:
    """Download an image from a URL and return a PIL Image."""
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img

def prepare_image(img: Image.Image, target_size=(200, 200)) -> Image.Image:
    """Ensure RGB mode and resize to target_size using nearest-neighbor."""
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

# 1. Download and resize
img = download_image(IMG_URL)
img = prepare_image(img, target_size=(200, 200))

# 2. Convert to NumPy and scale to [0, 1]
img_np = np.array(img).astype("float32") / 255.0  # shape: (H, W, C)

# 3. ImageNet normalization
mean = np.array([0.485, 0.456, 0.406], dtype="float32")
std  = np.array([0.229, 0.224, 0.225], dtype="float32")

img_norm = (img_np - mean) / std

print("Raw array shape (H, W, C):", img_np.shape)
print("First pixel AFTER normalization, R channel:", img_norm[0, 0, 0])

# 4. Convert to NCHW for ONNX (batch, channels, height, width)
x = np.transpose(img_norm, (2, 0, 1))  # (C, H, W)
x = np.expand_dims(x, axis=0)          # (1, C, H, W)
print("Model input shape:", x.shape)

Raw array shape (H, W, C): (200, 200, 3)
First pixel AFTER normalization, R channel: -1.073294
Model input shape: (1, 3, 200, 200)


## 3. Question 4 – Running inference with ONNX Runtime

Now we:

1. Create an `InferenceSession` with `hair_classifier_v1.onnx`.
2. Read the input and output node names.
3. Run the model on our preprocessed input `x`.
4. Print the scalar prediction value.

This scalar prediction value is what you use for **Question 4**.


In [6]:
import onnxruntime as ort

# Create ONNX Runtime session
session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])

# Get input and output names
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print("ONNX input name:", input_name)
print("ONNX output name:", output_name)

# Run inference
y_pred = session.run([output_name], {input_name: x})[0]

print("Raw model output array:", y_pred)
print("Scalar prediction:", float(y_pred[0][0]))

ONNX input name: input
ONNX output name: output
Raw model output array: [[0.08927477]]
Scalar prediction: 0.08927477151155472


## 4. Helper function – `predict_from_url`

To reuse the logic (e.g., for Lambda or other services), we wrap the whole pipeline into a single function.

It:

- Downloads the image
- Resizes to 200×200
- Scales to `[0, 1]`
- Normalizes with ImageNet stats
- Runs the ONNX model
- Returns a scalar float prediction


In [7]:
def predict_from_url(url: str) -> float:
    img = download_image(url)
    img = prepare_image(img, target_size=(200, 200))

    img_np = np.array(img).astype("float32") / 255.0
    img_norm = (img_np - mean) / std

    x = np.transpose(img_norm, (2, 0, 1))  # (C, H, W)
    x = np.expand_dims(x, axis=0)          # (1, C, H, W)

    y_pred = session.run([output_name], {input_name: x})[0]
    return float(y_pred[0][0])

# Quick sanity test
print("Test prediction from URL:", predict_from_url(IMG_URL))

Test prediction from URL: 0.08927477151155472


## 5. Lambda-style handler for serverless deployment

For AWS Lambda (with API Gateway), we typically use a handler with this signature:

```python
def lambda_handler(event, context):
    ...
```

The event often contains a JSON-encoded string in `event["body"]`.  
Here we expect a JSON object like:

```json
{"url": "<image_url>"}
```

The handler then:

1. Extracts the `url`.
2. Calls `predict_from_url(url)`.
3. Returns a JSON response with the prediction.


In [8]:
import json

def lambda_handler(event, context=None):
    """AWS Lambda handler that expects a JSON body with an image URL."""

    # If called via API Gateway, body is usually a JSON string
    if "body" in event and isinstance(event["body"], str):
        body = json.loads(event["body"])
    else:
        # Direct local invocation can pass the dict directly
        body = event

    url = body.get("url")
    if url is None:
        return {
            "statusCode": 400,
            "body": json.dumps({"error": "url is required"})
        }

    pred = predict_from_url(url)

    return {
        "statusCode": 200,
        "body": json.dumps({"prediction": pred})
    }

# Local test example (uncomment to run):
# test_event = {"body": json.dumps({"url": IMG_URL})}
# print(lambda_handler(test_event, None))

## 6. Docker & AWS Lambda – Questions 5 & 6

The course provides a pre-built Docker image:

- **Image name:** `agrigorev/model-2025-hairstyle:v1`
- It is based on an AWS Lambda Python image and already contains a model file: `hair_classifier_empty.onnx`.

### 6.1. Question 5 – Image size

1. Pull the image locally:

```bash
docker pull agrigorev/model-2025-hairstyle:v1
```

2. Check the image size:

```bash
docker images | grep model-2025-hairstyle
```

Use the value in the `SIZE` column as the answer to Question 5 (pick the closest among:
`88 Mb`, `208 Mb`, `608 Mb`, `1208 Mb`).

---

### 6.2. Extending the image with your Lambda code

Create a `Dockerfile` in the same directory as your `lambda_function.py` (which contains the handler and `predict_from_url`):


```Dockerfile
FROM agrigorev/model-2025-hairstyle:v1

# Install dependencies for preprocessing and ONNX inference
RUN pip install --no-cache-dir numpy pillow onnxruntime

# Copy your Lambda handler
COPY lambda_function.py .

# Entry point for AWS Lambda
CMD ["lambda_function.lambda_handler"]
```

Build and run the Docker image locally:

```bash
docker build -t hair-lambda .
docker run -p 8080:8080 hair-lambda
```

Then invoke it using the AWS Lambda Runtime Interface Emulator endpoint:

```bash
curl -X POST   -H "Content-Type: application/json"   -d '{"url": "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"}'   http://localhost:8080/2015-03-31/functions/function/invocations
```

The `prediction` value in the JSON response is what you use for **Question 6**
(choose the closest among: `-1.0`, `-0.10`, `0.10`, `1.0`).

---

### 6.3. Optional – Deploying to AWS

High-level steps (optional and not graded):

1. Push your Docker image to **ECR** (Elastic Container Registry).
2. Create a Lambda function using **container image**.
3. Increase its **memory** and **timeout** settings.
4. Test the Lambda function in the console.
5. Create an **API Gateway** endpoint to expose the Lambda over HTTPS.
